In [1]:
import requests


In [ ]:

r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
    "latitude": 30, "longitude": 115,
    "start_date": "2024-07-01", "end_date": "2024-08-31",
    "daily": "temperature_2m_max",
    "timezone": "Asia/Shanghai",
    "models": "era5_land",
})


In [3]:
r.raise_for_status()
data = r.json()
print(data["daily"]["time"][:3], data["daily"]["temperature_2m_max"][:3])

['2024-07-01', '2024-07-02', '2024-07-03'] [28.9, 28.5, 32.5]


In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"hourly": ["temperature_2m", "pressure_msl"],
}
responses = openmeteo.weather_api(url, params = params)

response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_pressure_msl = hourly.Variables(1).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["pressure_msl"] = hourly_pressure_msl

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)
